# 06. Scalability, Dask, Batching, and Memory

This notebook explains scaling behavior and uses repository benchmark artifacts to visualize runtime, memory, Dask batching, and a 10,000-individual run.

The central idea is simple: keep JAX as the fast leaf HMM engine, and let Dask schedule coarse tasks over `(variant block, sample batch, ploidy group)`. Do not split the HMM math into tiny Dask array operations.


In [1]:
from pathlib import Path
import os, sys, json, math, shutil, time

REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
sys.path.insert(0, str(REPO / 'src'))
FIG_DIR = REPO / 'docs' / 'tutorial_deep' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = REPO / 'benchmark_runs' / 'synth_5mb_2k_0p1x'
OUT_DIR = REPO / 'benchmark_runs' / 'tutorial_deep_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
print('repo:', REPO)
print('synthetic data exists:', DATA_DIR.exists())


repo: /home/bonnie/Documents/codex/STITCHV2
synthetic data exists: True


## Big-O Intuition

Let:

- `N` = samples
- `M` = variants
- `K` = founders
- `P` = ploidy
- `S` = hidden states

The dominant HMM memory is approximately proportional to:

```text
sample_batch_size * block_size * S
```

For fast diploid mode, `S = K^2`. For haploid, `S = K`. For generic polyploid, `S = choose(K + P - 1, P)`.

Total work scales roughly linearly in samples and variants once K/P are fixed:

```text
O(N * M * HMM_state_work)
```

The practical trick is not to hold all `N * M * S` cells in memory at once. Block SNPs and batch samples.


In [2]:
from stitchv2.dask_executor import estimate_hmm_task_memory_mb, plan_dask_chunks, polyploid_state_count

rows = []
for N in [48, 500, 1000, 5000, 10000]:
    for block in [500, 1000, 2000]:
        mem_all = estimate_hmm_task_memory_mb(n_samples=N, n_variants=block, n_founders=8, ploidy=2, return_genotype_posterior=True)
        plan = plan_dask_chunks(
            n_samples=N,
            n_variants=2000,
            n_founders=8,
            max_ploidy=2,
            configured_block_size=block,
            configured_sample_batch_size=0,
            target_task_memory_mb=512,
            min_block_size=128,
            min_sample_batch_size=64,
            return_genotype_posterior=True,
        )
        rows.append({'samples': N, 'configured_block': block, 'unbatched_task_mb': mem_all, **plan.to_dict()})
scale = pd.DataFrame(rows)
display(scale)

fig, ax = plt.subplots(figsize=(8, 4.5))
for block, sub in scale.groupby('configured_block'):
    ax.plot(sub['samples'], sub['unbatched_task_mb'], marker='o', label=f'unbatched block={block}')
ax.axhline(512, color='black', linestyle='--', linewidth=1, label='512 MB target')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('samples')
ax.set_ylabel('estimated one-task memory MB')
ax.set_title('Why sample batching is needed')
ax.legend()
fig.tight_layout()
out = FIG_DIR / '06_memory_scaling_unbatched.png'
fig.savefig(out, dpi=160)
print('wrote', out)


    samples  configured_block  unbatched_task_mb  block_size  sample_batch_size  estimated_task_memory_mb  state_count  \
0        48               500          23.803711         500                 48                 23.803711           64   
1        48              1000          47.607422        1000                 48                 47.607422           64   
2        48              2000          95.214844        2000                 48                 95.214844           64   
3       500               500         247.955322         500                500                247.955322           64   
4       500              1000         495.910645        1000                500                495.910645           64   
5       500              2000         991.821289        1000                500                495.910645           64   
6      1000               500         495.910645         500               1000                495.910645           64   
7      1000             

![Unbatched memory scaling](figures/06_memory_scaling_unbatched.png)


In [3]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for block, sub in scale.groupby('configured_block'):
    ax.plot(sub['samples'], sub['estimated_task_memory_mb'], marker='o', label=f'planned block={block}')
ax.axhline(512, color='black', linestyle='--', linewidth=1, label='512 MB target')
ax.set_xscale('log')
ax.set_xlabel('samples')
ax.set_ylabel('estimated planned task memory MB')
ax.set_title('Dask planner keeps leaf tasks within memory target')
ax.legend()
fig.tight_layout()
out = FIG_DIR / '06_memory_scaling_planned.png'
fig.savefig(out, dpi=160)
print('wrote', out)


wrote /home/bonnie/Documents/codex/STITCHV2/docs/tutorial_deep/figures/06_memory_scaling_planned.png


![Planned task memory](figures/06_memory_scaling_planned.png)

## Observed Serial vs Dask Benchmark

The repository contains a three-way synthetic benchmark for STITCH, STITCHV2 serial JAX, and STITCHV2-Dask. This was run on 48 samples and 2000 variants, which is small enough that Dask overhead can dominate. The value here is equality/diagnostics, not speedup.


In [4]:
dask_dir = REPO / 'benchmark_runs' / 'tutorial_deep_fresh_2026-05-01' / 'stitch_stitchv2_dask_fixed'
summary = json.loads((dask_dir / 'benchmark_summary.json').read_text())
runtime = pd.DataFrame(summary['runtime_memory'])
metrics = pd.DataFrame(summary['aggregate_metrics'])
per_snp = pd.read_parquet(dask_dir / 'per_snp_metrics_long.parquet')
print('Dask dashboard URL from benchmark:', summary.get('dask_dashboard_url'))
print('Dask performance report:', summary.get('dask_performance_report'))
display(runtime)
display(metrics)


Dask dashboard URL from benchmark: http://137.110.193.14:8786/status
Dask performance report: /home/bonnie/Documents/codex/STITCHV2/benchmark_runs/tutorial_deep_fresh_2026-05-01/stitch_stitchv2_dask_fixed/stitchv2_dask/dask_performance_report.html
          method                                            run_dir status  elapsed_seconds  raw_elapsed_seconds  dashboard_hold_seconds  \
0       STITCHV2  benchmark_runs/tutorial_deep_fresh_2026-05-01/...     ok         3.180798             3.180798                     0.0   
1  STITCHV2-Dask  benchmark_runs/tutorial_deep_fresh_2026-05-01/...     ok         3.639661             3.639661                     0.0   
2         STITCH  benchmark_runs/tutorial_deep_fresh_2026-05-01/...     ok         1.792764                  NaN                     NaN   

   peak_rss_mb  
0   539.238281  
1   681.882812  
2   126.667969  
          method        r2      rmse       mae  accuracy        f1  balanced_accuracy  info_mean  info_median  missing_rate

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
runtime.plot.bar(x='method', y='elapsed_seconds', ax=axes[0], legend=False)
axes[0].set_title('Runtime')
axes[0].set_ylabel('seconds')
runtime.plot.bar(x='method', y='peak_rss_mb', ax=axes[1], legend=False, color='tab:orange')
axes[1].set_title('Peak RSS')
axes[1].set_ylabel('MB')
for ax in axes:
    ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
out = FIG_DIR / '06_serial_dask_stitch_runtime_memory.png'
fig.savefig(out, dpi=160)
print('wrote', out)


wrote /home/bonnie/Documents/codex/STITCHV2/docs/tutorial_deep/figures/06_serial_dask_stitch_runtime_memory.png


In [6]:
metrics_to_plot = ['r2','accuracy','f1','balanced_accuracy','missing_rate','info']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
labels = sorted(per_snp['method'].unique())
for ax, metric in zip(axes, metrics_to_plot):
    sub = per_snp[['method', metric]].replace([np.inf, -np.inf], np.nan).dropna()
    labels_metric = [label for label in labels if not sub.loc[sub['method'] == label, metric].empty]
    data = [sub.loc[sub['method'] == label, metric].to_numpy() for label in labels_metric]
    if data:
        parts = ax.violinplot(data, showmeans=True, showextrema=False)
        for body in parts['bodies']:
            body.set_alpha(0.55)
    ax.set_xticks(range(1, len(labels_metric)+1))
    ax.set_xticklabels(labels_metric, rotation=30, ha='right')
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.25)
fig.suptitle('Per-SNP distributions: STITCH vs STITCHV2 vs STITCHV2-Dask', y=1.02)
fig.tight_layout()
out = FIG_DIR / '06_per_snp_violins_stitch_stitchv2_dask.png'
fig.savefig(out, dpi=160, bbox_inches='tight')
print('wrote', out)


wrote /home/bonnie/Documents/codex/STITCHV2/docs/tutorial_deep/figures/06_per_snp_violins_stitch_stitchv2_dask.png


![Runtime and memory](figures/06_serial_dask_stitch_runtime_memory.png)

![Per-SNP benchmark violins](figures/06_per_snp_violins_stitch_stitchv2_dask.png)


## Fresh 10,000-Individual Run

This notebook now uses a freshly rerun STITCHV2-Dask benchmark with exactly 10,000 individuals and 2,000 variants. It is not pulled from the older full-benchmark report artifacts.


In [7]:
scale10_path = REPO / 'benchmark_runs' / 'tutorial_deep_fresh_2026-05-01' / 'scalability_10k2k_dask' / 'scalability_10k2k_summary.json'
scale10 = json.loads(scale10_path.read_text())
summary10 = pd.DataFrame([{
    'method': 'STITCHV2-Dask fresh 10k x 2k',
    'samples': scale10['n_samples'],
    'variants': scale10['n_positions'],
    'elapsed_seconds': scale10['elapsed_seconds'],
    'peak_rss_mb_process': scale10['peak_rss_mb_process'],
    'memory_profile_peak_rss_mb': scale10['memory_profile_summary'].get('peak_rss_mb'),
    'memory_profile_peak_hmm_rss_mb': scale10['memory_profile_summary'].get('peak_hmm_rss_mb'),
    'dask_sample_batch_size': scale10['dask_sample_batch_size'],
    'block_size': scale10['block_size'],
}])
display(summary10)
print('Dask dashboard URL:', scale10.get('dask_run_summary', {}).get('dashboard_url'))
print('Dask performance report:', scale10.get('dask_run_summary', {}).get('performance_report'))

stage10 = pd.DataFrame(scale10['stage_timings'])
display(stage10[['block_id','seconds_read_extract','seconds_hmm','seconds_calibration','seconds_write','seconds_total','rss_mb_after_hmm','rss_mb_after_write']])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
summary10.plot.bar(x='method', y='elapsed_seconds', ax=axes[0], legend=False)
axes[0].set_title('Fresh 10k x 2k elapsed time')
axes[0].set_ylabel('seconds')
summary10.plot.bar(x='method', y='memory_profile_peak_rss_mb', ax=axes[1], legend=False, color='tab:green')
axes[1].set_title('Fresh 10k x 2k peak RSS')
axes[1].set_ylabel('MB')
for ax in axes:
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
out = FIG_DIR / '06_10k_observed_runtime_memory.png'
fig.savefig(out, dpi=160)
print('wrote', out)


                         method  samples  variants  elapsed_seconds  peak_rss_mb_process  memory_profile_peak_rss_mb  \
0  STITCHV2-Dask fresh 10k x 2k    10000      2000       147.356632          3848.179688                 1935.453125   

   memory_profile_peak_hmm_rss_mb  dask_sample_batch_size  block_size  
0                     1928.261719                     500        1000  
Dask dashboard URL: http://137.110.193.14:8786/status
Dask performance report: /home/bonnie/Documents/codex/STITCHV2/benchmark_runs/tutorial_deep_fresh_2026-05-01/scalability_10k2k_dask/run/dask_performance_report.html
   block_id  seconds_read_extract  seconds_hmm  seconds_calibration  seconds_write  seconds_total  rss_mb_after_hmm  rss_mb_after_write
0         0             28.899875    47.654438             0.000247       3.596446      80.151007       1439.121094         1643.527344
1         1              4.749144    54.955594             0.000351       3.531179      63.236268       1928.261719         

![Observed 10k run](figures/06_10k_observed_runtime_memory.png)

## Dask Dashboard and Report

Use this pattern when you want live diagnostics and a saved performance report:

```bash
stitchv2 run   --samples samples.parquet   --positions positions.parquet   --chromosome chr1   --output-dir runs/chr1_dask   --n-founders 8   --hmm-backend jax   --executor dask   --dask-n-workers 4   --dask-threads-per-worker 1   --dask-dashboard-address 127.0.0.1:8786   --dask-performance-report runs/chr1_dask/dask_report.html   --dask-task-stream runs/chr1_dask/task_stream.json
```

The dashboard URL is stored in `dask_run_summary.json`, and the HTML report can be opened after the run.


## Practical Scaling Guidance

- Increase `--block-size` until memory or compile time becomes uncomfortable.
- Set `--jax-sample-batch-size` or `--dask-sample-batch-size` when sample count is large.
- Use `--dask-target-task-memory-mb` to centralize chunk planning instead of hand-tuning every run.
- Keep Dask chunks coarse: variant blocks, sample batches, and ploidy groups. Tiny SNP-level tasks will be slower.
- For GPU, prefer one Dask worker per GPU and let each worker reuse the JAX compile cache for its shape.
- For 10,000 samples, do not run all samples in one JAX leaf task unless you have measured memory headroom.
